# M10 — canonical+LBS surfel avatar (v3) in the static map

Thin GPU runner for **`tracking/surfel_avatar.py` (v3)**. Logic lives in the fork; this notebook clones, installs, mounts Drive, sets paths, runs, displays.

v3 = the redesign after studying EfficientHuman / SGIA / GSSHuman (see `knowledge/papers.md` surfel-human section): appearance optimised **once in a canonical rest pose**, **LBS**-posed per frame (smplx), residual-rotation normals, trainable scale + **densification**, silhouette+**depth**+normal losses, MAMMA-jitter absorbed by `--smooth` + a pose-refine Δθ MLP. Replaces v1/v2 (one-surfel-per-vertex, shared colour → blur+holes).

Pre-filled for **Date03_Sub05_boxtiny**; run on an **A100** from the gmail account whose Drive holds the data. Every run writes to a **new run-tagged Drive dir** (never reuse an output dir).

**VERIFY on first run** (§6): (a) the posed body **lands on the person** (else the MAMMA calib convention / `--align_t`); (b) surfel disks **face the cameras**, not edge-on flecks (else the ΔR/normal frame is off); (c) densification **grows** the canonical count (printed per 500 iters — if it net-prunes, lower `--densify_grad_threshold` / raise `--opacity_cull` cautiously, cf. the deform-rig Lever-B gotcha).

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2+3. Mount Drive + clone the fork + build CUDA submodules + deps
Same stack as `smpl_person_render` (committed `render()` already takes the means/rotation overrides). `smplx` is required here — v3 poses **params** via LBS (it no longer renders MAMMA's exported `pred_vertices` mesh).

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os, sys
os.chdir('/content')
if not os.path.isdir('/content/gaussian-surfel-local-map'):
    !git clone --recursive https://github.com/steptang/gaussian-surfel-local-map.git
sys.path.insert(0, '/content/gaussian-surfel-local-map')
os.chdir('/content/gaussian-surfel-local-map')
!git pull -q   # need the branch with the v3 canonical+LBS tracking/surfel_avatar.py
!pip install -q plyfile open3d scikit-learn scipy mediapy imageio imageio-ffmpeg trimesh opencv-python-headless smplx
!pip install -q --no-build-isolation ./submodules/diff-surfel-rasterization
!pip install -q --no-build-isolation ./submodules/simple-knn
print('ready')

## 4. Paths (pre-filled for boxtiny; outputs → NEW run-tagged Drive dir)
- `MAMMA_ROOT` / `MAMMA_MODEL_DIR` — MAMMA exports + SMPL-X model dir on Drive (from `M10 - run_mamma_behave`).
- `SMPL_ROOT` — raw BEHAVE sequence (only needed for the optional GT run in §8).
- `RUN` — bump the tag (or let the auto-suffix find a free one) so every run keeps its outputs.

In [ ]:
DRIVE = '/content/drive/MyDrive/Research'
CONV_ROOT       = f'{DRIVE}/2dgs_output/datasets/behave_Date03_Sub05_boxtiny'
# NOTE the _cam2world suffix: mamma_exports/Date03_Sub05_boxtiny (no suffix) is the FIRST MAMMA
# attempt with the wrong world2cam calibration (scrambled world poses, params-only npz — burned
# two v3 runs). The good exports (cam2world calib + 'vertices' for rigid_correct) are:
MAMMA_ROOT      = f'{DRIVE}/datasets/mamma_exports/Date03_Sub05_boxtiny_cam2world'
MAMMA_MODEL_DIR = f'{DRIVE}/2dgs_output/mamma_run/body_models/smplx_locked_head'
SMPL_ROOT       = f'{DRIVE}/datasets/behave/sequences/Date03_Sub05_boxtiny'   # optional GT run
STATIC_PLY      = f'{DRIVE}/2dgs_output/results/smpl_person/static_boxtiny_7k.ply'  # recon cache: load if exists, else build+save (delete to force a rebuild)
N_SELECT        = 24

import os, glob
RUN = 'v3_6k_mamma'                      # <version>_<iters>_<modifier>; auto-suffixed if taken
base = f'{DRIVE}/2dgs_output/results/smpl_person/avatar/{RUN}'
OUT = base; k = 1
while os.path.isdir(OUT): k += 1; OUT = f'{base}_{k}'
os.makedirs(OUT)
assert os.path.isdir(CONV_ROOT), CONV_ROOT
assert os.path.isdir(MAMMA_ROOT), MAMMA_ROOT
assert os.path.isdir(MAMMA_MODEL_DIR), MAMMA_MODEL_DIR
# fail fast if the exports lack the posed mesh (rigid_correct needs it; params-only = the bad dir)
_probe = sorted(glob.glob(f'{MAMMA_ROOT}/frame_*/*.npz'))
assert _probe, f'no exports under {MAMMA_ROOT}'
import numpy as np
assert 'vertices' in np.load(_probe[0]), \
    f'{_probe[0]} has no "vertices" — params-only export; re-run behave_to_mamma output on the cam2world MAMMA run'
print('outputs ->', OUT)
print('static cache', 'HIT' if os.path.exists(STATIC_PLY) else 'MISS (will build+save)', '->', STATIC_PLY)
print('exports OK (vertices present):', len(_probe), 'frames')

## 5. Train the v3 avatar (MAMMA SMPL-X, full recipe ON)
~6k iters + a 7k-iter static map. All recipe pieces default ON; ablate with `--no_skin_mlp` / `--no_pose_refine` / `--smooth 0`. Watch the printed surfel count: it should **grow** through the densify window (500→4500).

In [ ]:
!python -m tracking.surfel_avatar \
  --conv_root "$CONV_ROOT" --smpl_root "$MAMMA_ROOT" --smpl_source mamma \
  --smpl_model_dir "$MAMMA_MODEL_DIR" --n_select $N_SELECT --view 0 \
  --static_ply "$STATIC_PLY" \
  --n_canon 60000 --iters 6000 --smooth 5 --out "$OUT"

## 6. Inspect + VERIFY
(a) body on the person? (b) disks facing the camera (crisp, not flecks)? (c) `canonical_surfels` in metrics > `--n_canon`? Then compare `person_psnr_mean` against v1/v2 (~run dirs `avatar/v1_*`, `avatar/v2_*`) and the deform baseline.

In [ ]:
import json
from IPython.display import Image, Video, display
m = json.load(open(f'{OUT}/metrics.json'))
print(json.dumps({k: m[k] for k in ('canonical_surfels','frames','iters','person_psnr_mean')}, indent=2))
display(Image(f'{OUT}/compare.png'))
display(Video(f'{OUT}/sequence.mp4', embed=True, width=720))

## 7. Score vs earlier avatar runs (v1/v2) + Part-A baselines
Pulls `person_psnr_mean` from every run dir under `avatar/` plus the Part A/A.5 mesh renders.

In [ ]:
import glob, json, os
rows = []
for d in sorted(glob.glob(f'{DRIVE}/2dgs_output/results/smpl_person/avatar/*')) + \
         sorted(glob.glob(f'{DRIVE}/2dgs_output/results/smpl_person/part*')):
    try:
        m = json.load(open(f'{d}/metrics.json'))
        rows.append((os.path.basename(d), m.get('person_psnr_mean'), m.get('canonical_surfels', m.get('surfels', '-'))))
    except FileNotFoundError:
        pass
print(f"{'run':28s} {'person-PSNR':>12s} {'surfels':>9s}")
for name, psnr, n in rows:
    print(f'{name:28s} {psnr:12.2f} {str(n):>9s}' if psnr is not None else f'{name:28s} {"-":>12s}')

## 8. (Optional) GT-SMPL variant — isolates the recipe from MAMMA jitter
Same avatar on BEHAVE's temporally-smooth SMPL-H fits. If GT is sharp and MAMMA isn't, the residual problem is pose jitter (raise `--smooth`, keep `--pose_refine`); if both are soft, it's the recipe/densification. Needs SMPL-H model files if BEHAVE pkl → smplx; put them under `MAMMA_MODEL_DIR`'s parent or point `--smpl_model_dir` elsewhere.

In [ ]:
OUT_GT = OUT + '_gt'
os.makedirs(OUT_GT, exist_ok=True)
!python -m tracking.surfel_avatar \
  --conv_root "$CONV_ROOT" --smpl_root "$SMPL_ROOT" --smpl_source gt \
  --smpl_model_dir "$MAMMA_MODEL_DIR" --n_select $N_SELECT --view 0 \
  --static_ply "$STATIC_PLY" \
  --n_canon 60000 --iters 6000 --smooth 0 --out "$OUT_GT"